# MedQA fine-tuning — Colab driver

This notebook **orchestrates only**. Every line of logic lives in `src/medqa/`, where
git can diff it and pytest can import it. If you need to change how something works,
edit the package on your Mac, push, and re-run the bootstrap cell — do not patch it here.

Runtime > Change runtime type > **T4 GPU** before you start.

In [ ]:
# ── Bootstrap: clone the repo and install ──
import os
REPO = "https://github.com/fayazhussain2821/llm-finetuning-medqa.git"
if not os.path.exists("/content/llm-finetuning-medqa"):
    !git clone -q {REPO} /content/llm-finetuning-medqa
%cd /content/llm-finetuning-medqa
!git pull -q                                   # always latest
!pip install -q -r requirements-colab.txt
!pip install -qe .
print("✅ ready — Runtime > Restart session, then continue from the next cell")

In [ ]:
# ── Hardware smoke test: fail here, not 40 minutes into training ──
import torch, transformers, peft, trl, datasets

assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > T4 GPU"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU {gpu.name} · {gpu.total_memory / 1024**3:.1f} GB · CUDA {torch.version.cuda}")
print("bf16 supported:", torch.cuda.is_bf16_supported())

# force a real GPU computation — a torch/driver mismatch surfaces HERE
(torch.randn(1000, device="cuda") @ torch.randn(1000, device="cuda")).item()
print("CUDA compute OK")

for lib in (torch, transformers, peft, trl, datasets):
    print(f"{lib.__name__:14s} {lib.__version__}")

In [ ]:
# ── Who are we pushing as? Set this BEFORE importing medqa.config ──
import os
os.environ["HF_USER"] = "Babblu2821"     # <- change if you forked this

from medqa import config, data, evaluate, models, train
print("adapters resolve to:", config.GPT2.hub_repo, "and", config.TINYLLAMA.hub_repo)

## Data

The dataset streams from the Hub — no Drive mount, no local copy. The split is seeded,
so both models are evaluated on exactly the same held-out rows.

In [ ]:
# ── Confirm MAX_LEN is still the right call for this dataset ──
raw = data.load_medquad()
print(raw)
print(data.probe_token_lengths(raw))

## Train both arms

Each call trains one arm and writes the adapter plus a `run_config.json` recording
exactly how it was made. Evaluation now runs *during* training, so an overfit shows up
in the logs while there is still time to stop.

In [ ]:
gpt2_metrics = train.train("gpt2", epochs=1)

In [ ]:
# free the GPT-2 training state before the 4-bit load
import gc, torch
gc.collect(); torch.cuda.empty_cache()

tinyllama_metrics = train.train("tinyllama", epochs=1)

## Measure

Four runs, not two. The `--base` runs are the **control condition**: the same base model
with no fine-tuning at all. Without them, nothing separates "QLoRA worked" from
"TinyLlama was already 9x bigger". They are forward passes only — no training, minutes not hours.

The headline number is **bits per byte**, not perplexity. GPT-2 and TinyLlama use different
tokenizers, so per-token perplexity puts the two models on different denominators.

In [ ]:
for key in ("gpt2", "tinyllama"):
    for base in (True, False):
        run_name, metrics = evaluate.evaluate_model(key, base=base)
        evaluate.write_metrics(run_name, metrics)

print(evaluate.comparison_table())

In [ ]:
# ── Qualitative side-by-side on held-out questions ──
import textwrap

gpt2_chat = models.DomainChatModel(config.GPT2, adapter=config.GPT2.output_dir)
tiny_chat = models.DomainChatModel(config.TINYLLAMA, adapter=config.TINYLLAMA.output_dir)

eval_ds = data.split_dataset(data.format_examples(data.load_medquad()))["test"]
short = lambda s: textwrap.shorten(s.replace("\n", " "), 380)

for ex in eval_ds.select(range(3)):
    print("=" * 90)
    print("Q:", ex["question"])
    print("-" * 90)
    print("GPT-2     :", short(gpt2_chat.generate(ex["question"])))
    print("TinyLlama :", short(tiny_chat.generate(ex["question"])))
    print("Reference :", short(ex["answer"]))

## Variance — is any of this bigger than the noise? (Phase 6.4)

Everything above is a **single training run**. A gap between two single runs is not
a result until you know how much a rerun would move it.

Two different noise sources, and they need different work:

* **Which questions were held out.** Measured locally, no GPU: `python -m medqa.variance`
  resamples the eval rows already scored.
* **Which training run happened** — data order, LoRA init, dropout. Only retraining
  measures this, which is why it lives here on the T4 and not on a laptop.

`--seed` moves the training run **and nothing else**: the split stays pinned to
`config.SPLIT_SEED`, so every seed is scored on identical held-out rows. Budget
roughly one extra training run per arm per seed.

In [ ]:
# Retrain each arm under the remaining seeds and score each run separately.
# Seeded runs are written to their own adapter dirs and their own metrics keys,
# so none of this overwrites the published seed-42 result.
import gc

import torch

from medqa import config, evaluate, train

for seed in config.TRAIN_SEEDS:
    if seed == config.TRAIN_SEED:
        continue  # the run trained above; already scored
    for key in ("gpt2", "tinyllama"):
        print(f"=== {key} · seed {seed} ===", flush=True)
        train.train(key, epochs=1, seed=seed)
        gc.collect()
        torch.cuda.empty_cache()

        run_name, metrics = evaluate.evaluate_model(key, seed=seed)
        evaluate.write_metrics(run_name, metrics)
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
from medqa import config, variance

# spread across seeds, per arm — the number Phase 6.4 exists to produce
for key in ("gpt2", "tinyllama"):
    spec = config.get_spec(key)
    runs = [spec.seeded_run_name(s) for s in config.TRAIN_SEEDS]
    spread = variance.seed_spread(runs)
    if spread["measured"]:
        print(f"{key}: mean {spread['mean']:.4f}  sd {spread['stdev']:.4f}  "
              f"range [{spread['min']:.4f}, {spread['max']:.4f}]  n={spread['n_runs']}")
    else:
        print(f"{key}: only {spread['n_runs']} run — train the other seeds first")

# and the eval-sampling intervals, which are a *different* question
print()
print(variance.text_table())

## Publish

Adapters go to the HF Hub, which is the only backup this project needs — they are 6 MB and
9 MB. The old Drive round-trip (`Loading Models.ipynb`) is retired: push by repo id here,
pull by repo id anywhere, including your Mac.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()          # paste a token with the **write** role

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

for spec in (config.GPT2, config.TINYLLAMA):
    api.create_repo(spec.hub_repo, repo_type="model", exist_ok=True)
    api.upload_folder(folder_path=str(spec.output_dir), repo_id=spec.hub_repo, repo_type="model")
    print(f"✅ {spec.output_dir.name} → {spec.hub_repo}")